# Step 3 — Across-trial peristimulus summary (opto)

**Input**: every `*_peristimulus_window.csv` produced by `step2_phantom_video_analysis_visualization_opto.ipynb` under `DATA_DIR`. Required columns: `time_s_rel_onset, opto_on, rescaled_intensity, wingbeat_amplitude, wingbeat_frequency`.

**What this notebook does**
1. Loads every peristimulus CSV.
2. Finds the common relative-time range across trials (intersection — `t_min = max(min)`, `t_max = min(max)`), interpolates each trial onto a uniform grid.
3. Computes mean ± SEM across trials for: stim (`opto_on`), wingbeat amplitude, wingbeat frequency, spiracle aperture (`rescaled_intensity`), inferred flight power (`inferred_flight_power`).
4. Saves a single summary CSV (grid × `[mean, sem, n]` per metric) + a 4-panel SVG/PNG figure.

**Stim alignment** — onset is at `t = 0`. The stim panel shows the across-trial mean of `opto_on` (= fraction of trials with opto on at each time). A red shaded band on every panel runs from 0 to the median per-trial offset, marking the typical stim window.

**Caveat** — averaging assumes all trials share a stim duration. If your folder mixes 3 s and 10 s opto trials, sort them into separate folders before running.

---

## 中文说明

**用途**：分析流程第三步——把 Step2A 产出的"单次实验刺激前后窗口"汇总到群体层面。它将多次光遗传实验对齐到刺激起点（onset = 0），计算跨实验的均值 ± 标准误，得到一张群体平均的刺激前后反应图，用于评估某条件下光遗传刺激对气门与飞行参数的平均效应。

**预期输入**：
- 文件类型：`DATA_DIR` 下由 Step2A 生成的每个 `*_peristimulus_window.csv`。这里只关注 `_peristimulus_window.csv` 这个后缀约定。
- 必需列：`time_s_rel_onset`（相对刺激起点的时间，0 = 起点）、`opto_on`（刺激是否开启的布尔列）、`rescaled_intensity`（气门开度）、`wingbeat_amplitude`（rad）、`wingbeat_frequency`（Hz）、`inferred_flight_power`（W/kg，由翅拍幅度与频率推算）。

**预期输出**（写入 `DATA_DIR`，前缀 `peristimulus_summary`）：
- `peristimulus_summary.csv`：每个网格点一行，含 `time_s_rel_onset` 及每个指标的 `[mean, sem, n]`。
- `peristimulus_summary.svg` / `.png`：多联图（刺激 + 翅拍幅度 + 翅拍频率 + 气门开度 + 飞行功率）。

**对齐方式**：以刺激起点为 0。刺激子图显示 `opto_on` 的跨实验均值（即每个时刻有多少比例的实验处于刺激态）。每个子图都用红色带从 0 标到"各实验偏移量的中位数"，表示典型的刺激窗口。

**注意**：求平均假设所有实验的刺激时长一致。若文件夹里混有 3 s 与 10 s 的实验，请先分到不同文件夹再运行。

### Block 1 — Imports & parameters

Sets `DATA_DIR` (folder of peristimulus CSVs from Step2A), the resampling grid resolution `N_GRID_HZ` (100 Hz), and the output filename prefix `OUT_PREFIX`.

**中文**：设置 `DATA_DIR`（Step2A 产出的刺激前后窗口 CSV 所在文件夹）、重采样网格分辨率 `N_GRID_HZ`（100 Hz）、以及输出文件名前缀 `OUT_PREFIX`。

In [1]:
# Block 1 — imports + parameters
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Folder to scan for peristimulus CSVs (output of opto-peristimulus single-trial notebook)
DATA_DIR  = Path(r"C:\Users\Lylah\Desktop\data_processing\SpMN4_ChR_3s")
RECURSIVE = False

# Resampling grid resolution (Hz). 100 Hz over a ~17 s window = ~1700 points — plenty for plotting.
N_GRID_HZ = 100.0

# Output filename prefix (in DATA_DIR)
OUT_PREFIX = "peristimulus_summary"

### Block 2 — Find & load peristimulus CSVs

`find_peristim_csvs` returns a sorted list of every `*_peristimulus_window.csv` in `DATA_DIR`; `load_peristim_csv` reads one and verifies the required columns are present.

**中文**：`find_peristim_csvs` 返回 `DATA_DIR` 中所有 `*_peristimulus_window.csv` 的有序列表；`load_peristim_csv` 读取单个文件并检查必需列是否齐全。

In [2]:
# Block 2 — find + load peristimulus CSVs

REQUIRED_COLS = [
    "time_s_rel_onset",
    "opto_on",
    "rescaled_intensity",
    "wingbeat_amplitude",
    "wingbeat_frequency",
    "inferred_flight_power",
]


def find_peristim_csvs(data_dir, recursive=False):
    """Return sorted list of *_peristimulus_window.csv under data_dir."""
    data_dir = Path(data_dir)
    it = (data_dir.rglob("*_peristimulus_window.csv") if recursive
          else data_dir.glob("*_peristimulus_window.csv"))
    return sorted(it)


def load_peristim_csv(csv_path):
    df = pd.read_csv(csv_path)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path.name} missing columns: {missing}")
    return df

### Block 3 — Interpolation helper

`interp_irregular` — the same NaN-safe interpolation helper used in Step 2: drops non-finite samples, sorts, de-duplicates timestamps, and returns NaN outside the data range. Used to place every trial on one common relative-time grid.

**中文**：`interp_irregular`——与 Step 2 相同的 NaN 安全插值函数：剔除非有限值、排序、去重时间戳，并在数据范围外返回 NaN。用于把每次实验对齐到同一公共的相对时间网格上。

In [3]:
# Block 3 — interp helper (Step 1 Block 6 pattern: NaN-safe, out-of-bounds -> NaN)

def interp_irregular(t, y, xq):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(t) & np.isfinite(y)
    t = t[m]
    y = y[m]
    if t.size == 0:
        return np.full(xq.shape, np.nan)
    order = np.argsort(t)
    t = t[order]
    y = y[order]
    uniq = np.concatenate([[True], np.diff(t) > 1e-15])
    t = t[uniq]
    y = y[uniq]
    if t.size < 2:
        return np.full(xq.shape, np.nan)
    return np.interp(xq, t, y, left=np.nan, right=np.nan)

### Block 4 — Stack trials onto a common relative-time grid

`stack_peristim_trials` builds the grid from the **intersection** of every trial's relative-time range (so each grid point is supported by all trials), interpolates each signal (including `opto_on` as 0/1) onto it, and returns the stacks plus the median per-trial stim offset (the last time `opto_on` is true, used to draw the typical stim window).

**中文**：`stack_peristim_trials` 用所有实验相对时间范围的**交集**来构建网格（保证每个网格点都有全部实验支撑），把每个信号（含按 0/1 处理的 `opto_on`）插值到该网格上，并返回堆叠数组以及各实验刺激偏移量的中位数（即 `opto_on` 最后为真的时刻，用于画出典型刺激窗口）。

In [4]:
# Block 4 — stack each trial onto a common relative-time grid

# Plot/CSV order: stim, amplitude, frequency, spiracle, power
METRICS = [
    ("opto_on",               "Opto on (fraction of trials)", ""),
    ("wingbeat_amplitude",    "Wingbeat amplitude",           "rad"),
    ("wingbeat_frequency",    "Wingbeat frequency",           "Hz"),
    ("rescaled_intensity",    "Spiracle aperture",            ""),
    ("inferred_flight_power", "Inferred flight power",        "W/kg"),
]


def stack_peristim_trials(csv_paths, n_grid_hz):
    """Return (t_grid, stacks_dict, trial_labels, median_offset_s).

    The grid spans the intersection of every trial's relative-time range, so
    each grid point is supported by every trial.
    """
    trials = []
    offsets_rel = []
    for p in csv_paths:
        try:
            df = load_peristim_csv(p)
        except Exception as e:
            print(f"[warn] {p.name}: {e} — skipping")
            continue
        if len(df) < 2:
            print(f"[warn] {p.name}: < 2 rows — skipping")
            continue
        # Per-trial offset_rel = last time where opto_on is True (since onset is at 0)
        opto_on = df["opto_on"].astype(bool).values
        if opto_on.any():
            offset_rel = float(df["time_s_rel_onset"].values[opto_on].max())
        else:
            offset_rel = np.nan
        trials.append((p, df))
        offsets_rel.append(offset_rel)

    if not trials:
        raise RuntimeError("No usable peristimulus CSVs found.")

    # Common time range = intersection of per-trial ranges
    t_min = max(float(df["time_s_rel_onset"].iloc[0])  for _, df in trials)
    t_max = min(float(df["time_s_rel_onset"].iloc[-1]) for _, df in trials)
    if not (t_max > t_min):
        raise RuntimeError(
            f"No common relative-time range across trials: t_min={t_min}, t_max={t_max}"
        )

    n_grid = int(round((t_max - t_min) * n_grid_hz)) + 1
    t_grid = np.linspace(t_min, t_max, n_grid)

    stacks = {col: [] for col, *_ in METRICS}
    trial_labels = []
    for p, df in trials:
        t = df["time_s_rel_onset"].values
        for col, *_ in METRICS:
            y = df[col].astype(float).values  # opto_on -> 0/1
            stacks[col].append(interp_irregular(t, y, t_grid))
        trial_labels.append(p.stem.replace("_peristimulus_window", ""))

    stacks = {k: np.asarray(v) for k, v in stacks.items()}
    median_offset = float(np.nanmedian(offsets_rel)) if offsets_rel else float("nan")
    return t_grid, stacks, trial_labels, median_offset

### Block 5 — Mean ± SEM

`mean_sem` — identical to Step 2: per-grid-point NaN-aware mean, SEM and effective n across trials.

**中文**：`mean_sem`——与 Step 2 完全一致：对每个网格点跨实验计算 NaN 安全的均值、标准误（SEM）和有效样本数 n。

In [5]:
# Block 5 — mean ± SEM (cite Step 1 Block 10)

def mean_sem(arr2d):
    if arr2d.size == 0:
        n_cols = arr2d.shape[1] if arr2d.ndim == 2 and arr2d.shape[1] > 0 else 0
        nan = np.full(n_cols, np.nan)
        return nan, nan, np.zeros(n_cols)
    mu = np.nanmean(arr2d, axis=0)
    n_eff = np.sum(np.isfinite(arr2d), axis=0).astype(float)
    sd = np.nanstd(arr2d, axis=0, ddof=1)
    sem = sd / np.sqrt(np.maximum(n_eff, 1.0))
    sem[~np.isfinite(sem)] = np.nan
    return mu, sem, n_eff

### Block 6 — Multi-panel mean ± SEM figure

`plot_peristim_summary` draws one panel per metric (the `opto_on` panel in red, the rest in blue), each with the mean line, a ± SEM band, a faint red stim window from 0 to the median offset, and dashed onset/offset markers. It honours the per-metric `YLIMS` (warning if the band is clipped) and annotates panel 0 with `n`. Saves SVG + PNG.

**中文**：`plot_peristim_summary` 为每个指标画一个子图（`opto_on` 用红色，其余用蓝色），每个子图含均值曲线、± SEM 阴影带、从 0 到偏移量中位数的淡红色刺激窗口，以及标出起止的虚线。它遵循每个指标的 `YLIMS`（阴影带被裁切则告警），并在第一个子图标注 `n`。保存为 SVG + PNG。

In [6]:
# Block 6 — 4-panel mean ± SEM figure

# y-axis limits per metric. None = autoscale. A clipping warning is printed at
# render time if the mean+/-SEM band falls outside the configured range.
YLIMS = {
    "rescaled_intensity":    (0.0, 1.0),
    "opto_on":               (-0.05, 1.10),
    "wingbeat_amplitude":    (3.14, 6.28),
    "wingbeat_frequency":    (100.0, 300.0),
    "inferred_flight_power": (100.0, 400.0),
}


def plot_peristim_summary(t_grid, stacks, median_offset, out_svg, out_png,
                          title=None, n_trials=None):
    fig, axes = plt.subplots(len(METRICS), 1, figsize=(14, 15), sharex=True)

    for ax, (col, label, unit) in zip(axes, METRICS):
        arr = stacks[col]
        # Light red shade for the typical stim window on every panel
        if np.isfinite(median_offset) and median_offset > 0:
            ax.axvspan(0.0, median_offset, color="red", alpha=0.10, zorder=0)
        if arr.shape[0] == 0:
            ax.text(0.5, 0.5, f"no trials for {col}",
                    transform=ax.transAxes, ha="center", va="center")
        else:
            mu, sem, _ = mean_sem(arr)
            line_color = "#d62728" if col == "opto_on" else "#1f77b4"
            ax.fill_between(t_grid, mu - sem, mu + sem,
                            color=line_color, alpha=0.25, lw=0)
            ax.plot(t_grid, mu, color=line_color, lw=1.2)
        # Onset / offset markers
        ax.axvline(0.0, color="k", linestyle="--", alpha=0.6, lw=0.8)
        if np.isfinite(median_offset):
            ax.axvline(median_offset, color="k", linestyle="--", alpha=0.6, lw=0.8)
        ylabel = f"{label} ({unit})" if unit else label
        ax.set_ylabel(ylabel)
        ax.grid(True, linestyle="--", alpha=0.35)
        ylim = YLIMS.get(col)
        if ylim is not None:
            ax.set_ylim(*ylim)
            if arr.shape[0] > 0:
                lo_data = float(np.nanmin(mu - sem))
                hi_data = float(np.nanmax(mu + sem))
                if lo_data < ylim[0] or hi_data > ylim[1]:
                    print(
                        f"[warn] {col}: mean+/-SEM range "
                        f"[{lo_data:.3f}, {hi_data:.3f}] is clipped by "
                        f"y-limits [{ylim[0]}, {ylim[1]}]"
                    )

    # n-trials annotation in upper-left of panel 0 (opto)
    if n_trials is None:
        # fall back to the largest stack as a sensible default
        n_trials = max((arr.shape[0] for arr in stacks.values()), default=0)
    axes[0].text(
        0.012, 0.88, f"n = {n_trials}",
        transform=axes[0].transAxes, ha="left", va="top", fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  edgecolor="black", alpha=0.85, linewidth=0.8),
    )

    axes[-1].set_xlabel("Time relative to opto onset (s)")
    axes[-1].set_xlim(t_grid[0], t_grid[-1])
    if title:
        fig.suptitle(title, fontsize=10)
        fig.subplots_adjust(top=0.96)
    plt.tight_layout()
    fig.savefig(out_svg, format="svg", bbox_inches="tight")
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close(fig)

### Block 7 — Save summary CSV

`save_summary_csv` writes one row per grid point: a `time_s_rel_onset` column plus `<metric>_mean`, `<metric>_sem`, `<metric>_n` for every signal — the tabular form of the Block 6 figure.

**中文**：`save_summary_csv` 每个网格点写一行：一列 `time_s_rel_onset`，外加每个信号的 `<metric>_mean`、`<metric>_sem`、`<metric>_n`——即 Block 6 图形的表格版本。

In [7]:
# Block 7 — save summary CSV (time_s_rel_onset + mean/sem/n per metric)

def save_summary_csv(t_grid, stacks, out_csv):
    out = pd.DataFrame({"time_s_rel_onset": t_grid})
    for col, *_ in METRICS:
        arr = stacks[col]
        mu, sem, n = mean_sem(arr)
        if arr.shape[0] == 0:
            mu = np.full(len(t_grid), np.nan)
            sem = np.full(len(t_grid), np.nan)
            n = np.zeros(len(t_grid))
        out[f"{col}_mean"] = mu
        out[f"{col}_sem"] = sem
        out[f"{col}_n"] = n.astype(int)
    out.to_csv(out_csv, index=False)

### Block 8 — Runner

The driver: finds the peristimulus CSVs, stacks them on the common grid, prints the window and median stim offset, then writes the summary CSV and the figure.

**中文**：主控单元：找到刺激前后窗口 CSV，在公共网格上堆叠，打印窗口范围与刺激偏移量中位数，然后写出汇总 CSV 与图形。

In [8]:
# Block 8 — runner

csvs = find_peristim_csvs(DATA_DIR, recursive=RECURSIVE)
print(f"Found {len(csvs)} peristimulus CSV(s) in {DATA_DIR}")
for p in csvs:
    print(f"  {p.name}")

t_grid, stacks, trial_labels, median_offset = stack_peristim_trials(csvs, N_GRID_HZ)
print(
    f"\nStacked {len(trial_labels)} trial(s); grid shape = {t_grid.shape}; "
    f"window = [{t_grid[0]:.3f}, {t_grid[-1]:.3f}] s; median stim offset = {median_offset:.3f} s"
)
for col, *_ in METRICS:
    print(f"  {col}: stack shape = {stacks[col].shape}")

out_csv = DATA_DIR / f"{OUT_PREFIX}.csv"
out_svg = DATA_DIR / f"{OUT_PREFIX}.svg"
out_png = DATA_DIR / f"{OUT_PREFIX}.png"

save_summary_csv(t_grid, stacks, out_csv)
plot_peristim_summary(
    t_grid, stacks, median_offset, out_svg, out_png,
    title=f"Peristimulus summary: n={len(trial_labels)} trials, median stim {median_offset:.2f} s",
    n_trials=len(trial_labels),
)

print(f"\nsaved: {out_csv}")
print(f"saved: {out_svg}")
print(f"saved: {out_png}")

Found 10 peristimulus CSV(s) in C:\Users\Lylah\Desktop\data_processing\SpMN4_ChR_3s
  2026_0401_142917_SpMN4_ChR_IS76603_ChR_7d_F_Fly4_Trial1_3000ms_thorax_sp1_ON_peristimulus_window.csv
  2026_0401_161356_SpMN4_ChR_IS76603_ChR_7d_F_Fly6_Trial1_3000ms_thorax_sp1_ON_peristimulus_window.csv
  2026_0403_142502_SpMN4_ChR_IS76603_ChR_7d_F_Fly1_Trial1_3000ms_thorax_sp1_ON_peristimulus_window.csv
  2026_0405_151757_SpMN4_ChR_IS76603_ChR_7d_F_Fly1_Trial1_3000ms_thorax_sp1_ON_peristimulus_window.csv
  2026_0422_143900_SpMN4_ChR_IS76603_ChR_2d_F_Fly1_Trial1_3000ms_thorax_sp1_ON_peristimulus_window.csv
  2026_0422_145226_SpMN4_ChR_IS76603_ChR_2d_F_Fly2_Trial1_3000ms_thorax_sp1_ON_peristimulus_window.csv
  2026_0422_152424_SpMN4_ChR_IS76603_ChR_2d_F_Fly3_Trial1_3000ms_thorax_sp1_ON_peristimulus_window.csv
  2026_0422_152927_SpMN4_ChR_IS76603_ChR_2d_F_Fly4_Trial1_3000ms_thorax_sp1_ON_peristimulus_window.csv
  2026_0427_100813_SpMN4_ChR_IS76603_ChR_4d_F_Fly1_Trial1_3000ms_thorax_sp1_ON_peristimulus_